In [4]:
import pandas as pd
import numpy as np

In [23]:
df= pd.read_parquet("../Data/raw/all_laps.parquet")
df.head()

,year,round,session,session_folder,event,country,location,event_date,Time,Driver,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:39:09.674000,HAM,...,False,Mercedes,0 days 00:37:09.970000,2021-03-28 15:07:09.980,124,2.0,False,,False,False
1,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:41:32.386000,HAM,...,False,Mercedes,0 days 00:39:09.674000,2021-03-28 15:09:09.684,4,2.0,False,,False,False
2,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:44:09.042000,HAM,...,False,Mercedes,0 days 00:41:32.386000,2021-03-28 15:11:32.396,41,2.0,False,,False,False
3,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:45:53.974000,HAM,...,False,Mercedes,0 days 00:44:09.042000,2021-03-28 15:14:09.052,126,2.0,False,,False,False
4,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,0 days 00:47:39.113000,HAM,...,False,Mercedes,0 days 00:45:53.974000,2021-03-28 15:15:53.984,671,2.0,False,,False,False


In [14]:
# Checking the columns in each df and what they represent
df.columns

Index(['year', 'round', 'session', 'session_folder', 'event', 'country',
       'location', 'event_date', 'Time', 'Driver', 'DriverNumber', 'LapTime',
       'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time',
       'Sector2Time', 'Sector3Time', 'Sector1SessionTime',
       'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2',
       'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife',
       'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus',
       'Position', 'Deleted', 'DeletedReason', 'FastF1Generated',
       'IsAccurate'],
      dtype='str')

The columns in this df can be grouped into 6 categories:
- *Race Identification* which tells us which race and session the data belongs to.
    - The columns in this category are `year`, `round`, `session`, `session_folder`, `event`, `country`, `location`, and `event_date`.

- *Driver and team* which tells us who dreve the lap and for which team.
    - The columns in this category are `Driver`, `DriverNumber`, and `Team`.

- *Lap and Timing* which tells us how long laps and individual sectors took.
    - The columns in this category are `Time`, `LapTime`, `LapNumber`, `Sector1Time`, `Sector2Time`, `Sector3Time`.

- *Pit stops and stints* which tells us tyre stints and pit-lane activity.
    - The columns in this category are `Stint`, `PitInTime`, `PitOutTime`.

- *Tyres* which tells us tyre type, age and whether or not the tyre was fresh.
    - The columns in this category are `Compound`, `TyreLife`, `FreshTyre`

- *Track and data quality* which tells us track conditions, performance, time stamps and data reliability.
    - The columns in this category are `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST`, `TrackStatus`, `Position`, `IsPersonalBest`, `Deleted`, `DeletedReason`, `FastF1Generated`, `IsAccurate`, `LapStartTime`, `LapStartDate`, `Sector1SessionTime`, `Sector2SessionTime`, `Sector3SessionTime`

A few of these columns are particularly important for this project;
- `LapTime` : Will be used to understand how lap pace changes.
- `Compound` : Will identify the tyre being used (soft, hard, medium, etc).
- `TyreLife` : Will identify how old the tyres are in laps.
- `Stint` : Will help distinguish different tyre stints during a race.
- `PitInTime` and `PitOutTime` : Will be used to identify pit-stop activity.
- `TrackStatus` : Will help identify changing track conditions.
- `Position` : Will help understand the drivers race position.

The `Time` column is the timestamp of the lap whereas the `LapTime` column is the actual duration of the lap.
- `Time` column tells us how long it took for a lap to end since the session begun. We use the term session instead of race because in F1 that time could be for practice, qualifying or a race.
- `LapTime` column tells us how long the lap lasted.

In [ ]:
# Inspecting the data types in each column
df.dtypes

year                            int64
round                           int64
session                           str
session_folder                    str
event                             str
country                           str
location                          str
event_date             datetime64[ns]
Time                  timedelta64[ns]
Driver                            str
DriverNumber                      str
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time           timedelta64[ns]
Sector1SessionTime    timedelta64[ns]
Sector2SessionTime    timedelta64[ns]
Sector3SessionTime    timedelta64[ns]
SpeedI1                       float64
SpeedI2                       float64
SpeedFL                       float64
SpeedST                       float64
IsPersonalBe

In the df, the data types contained in each of the column are as follows;
- int64 : `year`, `round`
- float64 : `LapNumber`, `Stint`, `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST`, `TireLife`, `Position`

- string : `session`, `session_folder`, `event`, `country`, `location`, `Driver`, `DriverNumber`, `Compound`, `Team`, `TrackStatus`, `DeletedReason`

- datetime64[ns] : `event_date`, `LapStartDate`
- timedelta64[ns] : `Time`, `LapTime`, `PitOutTime`, `PitInTime`, `Sector1Time`, `Sector2Time`, `Sector3Time`, `Sector1SessionTime`, `Sector2SessionTime`, `Sector3SessionTime`,  `LapStartTime`

- bool : `FreshTyre`, `Deleted`, `FastF1Generated`, `IsAccurate`

- object : `IsPersonalBest`

In [16]:
# Checking for missing values
df.isna().sum()

year                       0
round                      0
session                    0
session_folder             0
event                      0
country                    0
location                   0
event_date                 0
Time                       0
Driver                     0
DriverNumber               0
LapTime                  526
LapNumber                  0
Stint                    354
PitOutTime            120754
PitInTime             120731
Sector1Time             2660
Sector2Time              202
Sector3Time              319
Sector1SessionTime      2931
Sector2SessionTime       202
Sector3SessionTime       319
SpeedI1                  161
SpeedI2                 1048
SpeedFL                 4473
SpeedST                  217
IsPersonalBest           113
Compound                   0
TyreLife                 989
FreshTyre                  0
Team                       0
LapStartTime               0
LapStartDate            1057
TrackStatus                0
Position      

In the df, the only columns with missing values are;
- `LapTime` with `526`
- `Stint` with `354`
- `PitOutTime` with `120,754`
- `PitInTime` with `120,731`
- `Sector1Time` with `2,660`
- `Sector2Time` with `202`
- `Sector3Time` with `319`
- `Sector1SessionTime` with `2,931`
- `Sector2SessionTime` with `202`
- `Sector3SessionTime` with `319`
- `SpeedI1` with `161`
- `SpeedI2` with `1,048`
- `SpeedFL` with `4,473`
- `SpeedST` with `217`
- `IsPersonalBest` with `113`
- `TyreLife` with `989`
- `LapStartDate` with `1,057`
- `Position` with `160`
- `DeletedReason` with `113`

All the other columns have no missing values.

In [ ]:
# Checking for duplicates
df.duplicated().sum()

0         False
1         False
2         False
3         False
4         False
          ...  
125041    False
125042    False
125043    False
125044    False
125045    False
Length: 125046, dtype: bool

There are not duplicate rows in the entire df.